In [9]:
import pandas as pd

df = pd.read_csv('PS_20174392719_1491204439457_log.csv')
print(df.shape)
print(df.head())

(6362620, 11)
   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  


In [10]:
# encoding 'type'
df['type_encoded'] = pd.factorize(df['type'])[0]

In [11]:
df = df.drop(['nameOrig', 'nameDest', 'step', 'type', 'isFlaggedFraud'], axis=1)

In [12]:
from sklearn.model_selection import train_test_split

X = df.drop('isFraud', axis=1)  # Features (everything except target)
y = df['isFraud']                # Target (what we're predicting)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 80% train, 20% test
    random_state=42,    # For reproducibility
    stratify=y          # Keep same fraud percentage in train & test
)

In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert back to DataFrame
X_train = pd.DataFrame(X_train, columns=X.columns)
X_test = pd.DataFrame(X_test, columns=X.columns)

In [14]:
from sklearn.ensemble import RandomForestClassifier

# Create model
model = RandomForestClassifier(
    n_estimators=100,      # 100 trees
    class_weight='balanced', # Handle imbalance (NO SMOTE NEEDED!)
    random_state=42
)

# Train it
model.fit(X_train, y_train)

print("✓ Model trained!")

✓ Model trained!


In [15]:
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]  # Probability of fraud

In [16]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix, classification_report
)

# Precision: Of predicted frauds, how many are correct?
precision = precision_score(y_test, y_pred)
print(f"Precision: {precision:.3f}")

# Recall: Of all real frauds, how many did we catch?
recall = recall_score(y_test, y_pred)
print(f"Recall: {recall:.3f}")

# F1-Score: Balance of precision and recall
f1 = f1_score(y_test, y_pred)
print(f"F1-Score: {f1:.3f}")

# ROC-AUC: Overall model performance (0-1, higher is better)
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC: {roc_auc:.3f}")

# Show detailed breakdown
print("\n" + classification_report(y_test, y_pred))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Precision: 0.963
Recall: 0.790
F1-Score: 0.868
ROC-AUC: 0.997

              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.96      0.79      0.87      1643

    accuracy                           1.00   1272524
   macro avg       0.98      0.89      0.93   1272524
weighted avg       1.00      1.00      1.00   1272524


Confusion Matrix:
[[1270831      50]
 [    345    1298]]
